# Connect to PostgreSQL

In [18]:
import psycopg2

try:
   # Connect to PostgreSQL
   connection = psycopg2.connect(
       dbname="Evolution",
       user="ev",
       password="Temp@123",
       host="192.168.4.51", # or your server's IP address
       port="5432" # default PostgreSQL port
   )
   print("Connection established successfully!")
except Exception as e:
   print(f"Error: {e}")


Connection established successfully!


In [19]:
cur = connection.cursor()

# Load the data

In [20]:
import pandas as pd
import os
import json


In [30]:
directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"


In [21]:
table_columns_path = 'table_columns.json'
with open(table_columns_path, 'r', encoding='utf-8-sig') as file:
                table_columns = json.load(file)

In [5]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)

In [22]:
def pre_row(row, table_name):
    for col in table_columns[table_name]:
        if col not in row or row[col] == '-' or row[col] == '' or row[col] == ' ':
            
            if table_columns[table_name][col] == "int" or table_columns[table_name][col] == "float":
                    row[col] = ' 0 '
            else:
                 row[col] = ' Null '
    return row

In [23]:
def row2text():

	return eval(text_data)
        

# Load the embedding model

In [24]:
from langchain_ollama import OllamaEmbeddings

emb = OllamaEmbeddings(model="bge-large", base_url="http://192.168.43.220:11435")

def generate_embedding(text): 
    text = str(text)
    embedding = emb.embed_query(text)
    return embedding

In [45]:
# Delete the table
x = cur.execute("""DROP TABLE PaymentLinks """)

connection.commit()
connection.rollback()

In [44]:

connection.commit()
connection.rollback()

# Payments

In [14]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS Payments (
    Id                               INTEGER PRIMARY KEY,
    PaymentReceiptDetailsName        TEXT,
    Status                           TEXT,
    Amount                           DOUBLE PRECISION,
    Account                          TEXT,
    PaymentMethod                    TEXT,
    DateOfReceipt                    TEXT,
    ChqDDTTNumber                    TEXT,
    DateOfTransaction                TEXT,
    ChequeDepositDate                TEXT,
    ChequeReturnedDate               TEXT,
    DealId                           INTEGER,
    ReceivedByEVDate                 TEXT,
    TransferredToDeveloperDate       TEXT,
    ReceivedDate                     TEXT,
    POPReference                     INTEGER,
    isFormRequested                  BOOLEAN,
    IsLoading                        BOOLEAN,
    Currency                         TEXT,
    CustomerId                       INTEGER,
    PaymentStatus                    TEXT,
    CreatedBy                        TEXT,
    CreationDate                     TEXT,
    IsDeleted                        BOOLEAN,
    -- embed_* columns
    embed_PaymentReceiptDetailsName  VECTOR(1024),
    embed_Status                     VECTOR(1024),
    embed_Account                    VECTOR(1024),
    embed_PaymentMethod              VECTOR(1024),
    embed_PaymentStatus              VECTOR(1024),
    row_text                         TEXT,
    embedding                        VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

directory = "C:/Users/maryam.maksour/OneDrive - AL BAYARI/Desktop/RAG/DATA"

full_path = os.path.join(directory, "Payments.json")
file_name = "Payments"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                if i <= 3600:
                      i+=1
                      continue
                row2 = pre_row(row, file_name)

                # Build row_text
                txt = row2text()

                # Main embedding
                embedding = generate_embedding(txt)

                # Per-field embeddings (safe None checks)
                e_receipt_name  = generate_embedding(row2['PaymentReceiptDetailsName']) if row2.get('PaymentReceiptDetailsName') else None
                e_status        = generate_embedding(row2['Status']) if row2.get('Status') else None
                e_account       = generate_embedding(row2['Account']) if row2.get('Account') else None
                e_method        = generate_embedding(row2['PaymentMethod']) if row2.get('PaymentMethod') else None
                e_pay_status    = generate_embedding(row2['PaymentStatus']) if row2.get('PaymentStatus') else None

                cur.execute("""
                    INSERT INTO Payments (
                        Id, PaymentReceiptDetailsName, Status, Amount, Account, PaymentMethod,
                        DateOfReceipt, ChqDDTTNumber, DateOfTransaction, ChequeDepositDate, ChequeReturnedDate,
                        DealId, ReceivedByEVDate, TransferredToDeveloperDate, ReceivedDate, POPReference,
                        isFormRequested, IsLoading, Currency, CustomerId, PaymentStatus, CreatedBy, CreationDate,
                        IsDeleted, embed_PaymentReceiptDetailsName, embed_Status, embed_Account, embed_PaymentMethod,
                        embed_PaymentStatus, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], row2['PaymentReceiptDetailsName'], row2['Status'], float(row2['Amount']),
                    row2['Account'], row2['PaymentMethod'], row2['DateOfReceipt'], row2['ChqDDTTNumber'],
                    row2['DateOfTransaction'], row2['ChequeDepositDate'], row2['ChequeReturnedDate'],
                    int(row2['DealId']), row2['ReceivedByEVDate'], row2['TransferredToDeveloperDate'],
                    row2['ReceivedDate'], int(row2['POPReference']), bool(row2['isFormRequested']), bool(row2['IsLoading']),
                    row2['Currency'], int(row2['CustomerId']), row2['PaymentStatus'], row2['CreatedBy'],
                    row2['CreationDate'], bool(row2['IsDeleted']),
                    e_receipt_name, e_status, e_account, e_method, e_pay_status,
                    txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)


# InstallmentPlans

In [15]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                


x = cur.execute("""
CREATE TABLE IF NOT EXISTS InstallmentPlans (
    Id                INTEGER PRIMARY KEY,
    Name              TEXT,
    Priority          INTEGER,
    Percentage        DOUBLE PRECISION,
    Amount            DOUBLE PRECISION,
    DueDate           TEXT,
    PaymentDate       TEXT,
    Settled           BOOLEAN,
    Status            TEXT,
    PaidAmount        DOUBLE PRECISION,
    RemainingAmount   DOUBLE PRECISION,
    DealId            INTEGER,
    AccountType       TEXT,
    CreatedBy         TEXT,
    CreationDate      TEXT,
    IsDeleted         BOOLEAN,
    -- embed_* columns
    embed_Name        VECTOR(1024),
    embed_Status      VECTOR(1024),
    embed_AccountType VECTOR(1024),
    row_text          TEXT,
    embedding         VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "InstallmentPlans.json")
file_name = "InstallmentPlans"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                print(txt)
                embedding = generate_embedding(txt)

                e_name        = generate_embedding(row2['Name']) if row2.get('Name') else None
                e_status      = generate_embedding(row2['Status']) if row2.get('Status') else None
                e_accounttype = generate_embedding(row2['AccountType']) if row2.get('AccountType') else None

                cur.execute("""
                    INSERT INTO InstallmentPlans (
                        Id, Name, Priority, Percentage, Amount, DueDate, PaymentDate, Settled, Status,
                        PaidAmount, RemainingAmount, DealId, AccountType, CreatedBy, CreationDate,
                        IsDeleted, embed_Name, embed_Status, embed_AccountType, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], row2['Name'], int(row2['Priority']), float(row2['Percentage']),
                    float(row2['Amount']), row2['DueDate'], row2['PaymentDate'], bool(row2['Settled']),
                    row2['Status'], float(row2['PaidAmount']), float(row2['RemainingAmount']),
                    int(row2['DealId']), row2['AccountType'], row2['CreatedBy'], row2['CreationDate'],
                    bool(row2['IsDeleted']), e_name, e_status, e_accounttype, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)

 Installment plan with id 1: name 'Admin Fee'. priority 1. percentage 0.0%. amount 5250.0. due on 2025-07-15T00:00:00, paid on 2025-06-04T00:00:00. settled: True. status Active. paid 5250.0, remaining 0.0. deal id 1. account type Corporate Account Of Developer. created by  Null  on 2025-07-23T05:13:06.5688430. is deleted: False. 
 Installment plan with id 2: name 'DLD Fee'. priority 2. percentage 4.0%. amount 119431.08. due on 2025-07-15T00:00:00, paid on 2025-06-04T00:00:00. settled: False. status Active. paid 14750.0, remaining 104681.08. deal id 1. account type Corporate Account Of Developer. created by  Null  on 2025-07-23T05:13:06.5706550. is deleted: False. 
 Installment plan with id 3: name 'Upon Signing SPA'. priority 3. percentage 20.0%. amount 597155.4. due on 2025-07-15T00:00:00, paid on  Null . settled: False. status Active. paid 0.0, remaining 597155.4. deal id 1. account type Escrow Account of Developer. created by  Null  on 2025-07-23T05:13:06.5706570. is deleted: False.

# InstalmentAdjusted

In [ ]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS InstalmentAdjusted (
    Id                INTEGER PRIMARY KEY,
    InstalmentType    TEXT,
    Amount            DOUBLE PRECISION,
    PaymentMethod     TEXT,
    PaymentId         INTEGER,
    InstalmentPlanId  INTEGER,
    AccountType       TEXT,
    Currency          TEXT,
    CreatedBy         TEXT,
    CreationDate      TEXT,
    IsDeleted         BOOLEAN,
    -- embed_* columns
    embed_InstalmentType VECTOR(1024),
    embed_PaymentMethod   VECTOR(1024),
    embed_AccountType     VECTOR(1024),
    row_text              TEXT,
    embedding             VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "InstalmentAdjusted.json")
file_name = "InstalmentAdjusted"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
              
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                e_type       = generate_embedding(row2['InstalmentType']) if row2.get('InstalmentType') else None
                e_method     = generate_embedding(row2['PaymentMethod']) if row2.get('PaymentMethod') else None
                e_account    = generate_embedding(row2['AccountType']) if row2.get('AccountType') else None

                cur.execute("""
                    INSERT INTO InstalmentAdjusted (
                        Id, InstalmentType, Amount, PaymentMethod, PaymentId, InstalmentPlanId,
                        AccountType, Currency, CreatedBy, CreationDate, IsDeleted,
                        embed_InstalmentType, embed_PaymentMethod, embed_AccountType, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], row2['InstalmentType'], float(row2['Amount']), row2['PaymentMethod'],
                    int(row2['PaymentId']), int(row2['InstalmentPlanId']), row2['AccountType'],
                    row2['Currency'], row2['CreatedBy'], row2['CreationDate'], bool(row2['IsDeleted']),
                    e_type, e_method, e_account, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)

# OnlinePaymentTransactions


In [40]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS OnlinePaymentTransactions (
    Id                  INTEGER PRIMARY KEY,
    TransactionNumber   TEXT,
    Amount              DOUBLE PRECISION,
    Status              TEXT,
    PaymentLink         TEXT,
    CreationDateTime    TEXT,
    BusinessEntityId    INTEGER,
    BusinessEntityType  INTEGER,
    embed_status        VECTOR(1024),
    row_text            TEXT,
    embedding           VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "OnlinePaymentTransactions.json")
file_name = "OnlinePaymentTransactions"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                print(txt)
                embedding = generate_embedding(txt)

                e_status = generate_embedding(row2['Status'])  

                cur.execute("""
                    INSERT INTO OnlinePaymentTransactions (
                        Id, TransactionNumber, Amount, Status, PaymentLink, CreationDateTime,
                        BusinessEntityId, BusinessEntityType, embed_status, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], row2['TransactionNumber'], float(row2['Amount']),
                    row2['Status'], row2['PaymentLink'], row2['CreationDateTime'],
                    int(row2['BusinessEntityId']), int(row2['BusinessEntityType']),
                    e_status, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)

 Online payment transaction with id 1: transaction no. 87cb87ce-3795-441f-b94c-ed45d4021a1e. amount 150000.0. status OrderCreate. link https://secure.ccavenue.ae/transaction/transaction.do?command=initiateTransaction&encRequest=de9e8ffc3ec57c449075b6cfde5f3849dbf371b7fded7d4503a560e1934484d9fc1a8395f806529807c83691bc8bf3874085c6c89e858a23b4c7f760c569fe81c195b4e42e7c633e4101018f4fd889a9edd1fe464a0a75490fec79510cda378f09a01a994d848f6c606b9f11bc2460f8a1e907881aba1d3a361437505c2b76c42bca742728039f15a0cccf4afe71f26a7ba844d514c566fe5e4d5301622752428f5320e6f92b9045e399edd06e0db6149f7166ebdb30f470130867e108854683e1b3d7f3dc011d024e508b08fc0b645504c159850238b0954b64b95452515454a1247e7112266febde0129dbd84e562f9e8232428bdc1cbaecf6c69cf49a12d2&access_code=AVCX05MG42AR98XCRA. created on 0001-01-01T00:00:00. business entity id 3, type 1. 
OnlinePaymentTransactions
duplicate key value violates unique constraint "onlinepaymenttransactions_pkey"
DETAIL:  Key (id)=(1) already exists.



# PaymentLinks

In [46]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS PaymentLinks (
    Id                      INTEGER PRIMARY KEY,
    Amount                  DOUBLE PRECISION,
    Currency                TEXT,
    PaymentType             TEXT,
    PaymentUrl              TEXT,
    Status                  TEXT,
    ExpiryDate              TEXT,
    RefNo                   TEXT,
    PaymentLinkReferenceId  TEXT,
    ProjectId               INTEGER,
    AgentId                 INTEGER,
    CreatedBy               TEXT,
    CreationDate            TEXT,
    IsDeleted               BOOLEAN,
    IsSplited               BOOLEAN,
    RemainingAmount         DOUBLE PRECISION,
    ServiceChargePercentage DOUBLE PRECISION,
    -- embed_* columns
    embed_PaymentType       VECTOR(1024),
    embed_Status            VECTOR(1024),
    row_text                TEXT,
    embedding               VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "PaymentLinks.json")
file_name = "PaymentLinks"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                e_type   = generate_embedding(row2['PaymentType']) if row2.get('PaymentType') else None
                e_status = generate_embedding(row2['Status']) if row2.get('Status') else None

                cur.execute("""
                    INSERT INTO PaymentLinks (
                        Id, Amount, Currency, PaymentType, PaymentUrl, Status, ExpiryDate,
                        RefNo, PaymentLinkReferenceId, ProjectId, AgentId, CreatedBy, CreationDate,
                        IsDeleted, IsSplited, RemainingAmount, ServiceChargePercentage,
                        embed_PaymentType, embed_Status, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], float(row2['Amount']), row2['Currency'], row2['PaymentType'],
                    row2['PaymentUrl'], row2['Status'], row2['ExpiryDate'], row2['RefNo'],
                    row2['PaymentLinkReferenceId'], int(row2['ProjectId']), int(row2['AgentId']),
                    row2['CreatedBy'], row2['CreationDate'], bool(row2['IsDeleted']), bool(row2['IsSplited']),
                    float(row2['RemainingAmount']), float(row2['ServiceChargePercentage']),
                    e_type, e_status, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)

# PaymentPlanDetails

In [47]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS PaymentPlanDetails (
    Id                 INTEGER PRIMARY KEY,
    SalesProposalId    INTEGER,
    EVInstalment       TEXT,
    DueDate            TEXT,
    Amount             DOUBLE PRECISION,
    Percentage         DOUBLE PRECISION,
    PaymentTermItemId  INTEGER,
    CreatedBy          TEXT,
    CreationDate       TEXT,
    IsDeleted          BOOLEAN,
    -- embed_* columns
    embed_EVInstalment VECTOR(1024),
    row_text           TEXT,
    embedding          VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "PaymentPlanDetails.json")
file_name = "PaymentPlanDetails"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                e_inst = generate_embedding(row2['EVInstalment']) if row2.get('EVInstalment') else None

                cur.execute("""
                    INSERT INTO PaymentPlanDetails (
                        Id, SalesProposalId, EVInstalment, DueDate, Amount, Percentage,
                        PaymentTermItemId, CreatedBy, CreationDate, IsDeleted, embed_EVInstalment, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], int(row2['SalesProposalId']), row2['EVInstalment'],
                    row2['DueDate'], float(row2['Amount']), float(row2['Percentage']),
                    int(row2['PaymentTermItemId']), row2['CreatedBy'], row2['CreationDate'],
                    bool(row2['IsDeleted']), e_inst, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)


# PaymentSplits

In [48]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS PaymentSplits (
    Id                    INTEGER PRIMARY KEY,
    PaymentLinkId         TEXT,
    Amount                DOUBLE PRECISION,
    Status                TEXT,
    IsFullPaymentAmount   BOOLEAN,
    CreatedBy             TEXT,
    CreationDate          TEXT,
    IsDeleted             BOOLEAN,
    -- embed_* columns
    embed_Status          VECTOR(1024),
    row_text              TEXT,
    embedding             VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "PaymentSplits.json")
file_name = "PaymentSplits"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                e_status = generate_embedding(row2['Status']) if row2.get('Status') else None

                cur.execute("""
                    INSERT INTO PaymentSplits (
                        Id, PaymentLinkId, Amount, Status, IsFullPaymentAmount,
                        CreatedBy, CreationDate, IsDeleted, embed_Status, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], int(row2['PaymentLinkId']), float(row2['Amount']),
                    row2['Status'], bool(row2['IsFullPaymentAmount']), row2['CreatedBy'],
                    row2['CreationDate'], bool(row2['IsDeleted']), e_status, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)

# PaymentTerms

In [49]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS PaymentTerms (
    Id            INTEGER PRIMARY KEY,
    PTID          TEXT,
    AdminFee      DOUBLE PRECISION,
    DLDFee        DOUBLE PRECISION,
    Name          TEXT,
    ExchangeRate  DOUBLE PRECISION,
    Status        BOOLEAN,
    Currency      TEXT,
    BedType       TEXT,
    CreatedBy     TEXT,
    CreationDate  TEXT,
    IsDeleted     BOOLEAN,
    -- embed_* columns
    embed_Name     VECTOR(1024),
    embed_Status   VECTOR(1024),
    embed_BedType  VECTOR(1024),
    row_text       TEXT,
    embedding      VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "PaymentTerms.json")
file_name = "PaymentTerms"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                e_name    = generate_embedding(row2['Name']) if row2.get('Name') else None
                e_status  = generate_embedding(str(row2['Status'])) if 'Status' in row2 else None  # boolean → str
                e_bed     = generate_embedding(row2['BedType']) if row2.get('BedType') else None

                cur.execute("""
                    INSERT INTO PaymentTerms (
                        Id, PTID, AdminFee, DLDFee, Name, ExchangeRate, Status,
                        Currency, BedType, CreatedBy, CreationDate, IsDeleted,
                        embed_Name, embed_Status, embed_BedType, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], row2['PTID'], float(row2['AdminFee']), float(row2['DLDFee']),
                    row2['Name'], float(row2['ExchangeRate']), bool(row2['Status']),
                    row2['Currency'], row2['BedType'], row2['CreatedBy'], row2['CreationDate'],
                    bool(row2['IsDeleted']), e_name, e_status, e_bed, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)
 

# PaymentTermItems

In [50]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS PaymentTermItems (
    Id               INTEGER PRIMARY KEY,
    Name             TEXT,
    Gap              INTEGER,
    Percentage       DOUBLE PRECISION,
    DueDate          TEXT,
    PaymentTermId    INTEGER,
    MilestoneGapType TEXT,
    CreatedBy        TEXT,
    CreationDate     TEXT,
    IsDeleted        BOOLEAN,
    -- embed_* columns
    embed_Name           VECTOR(1024),
    embed_MilestoneGapType VECTOR(1024),
    row_text            TEXT,
    embedding           VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "PaymentTermItems.json")
file_name = "PaymentTermItems"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                e_name   = generate_embedding(row2['Name']) if row2.get('Name') else None
                e_gaptyp = generate_embedding(row2['MilestoneGapType']) if row2.get('MilestoneGapType') else None

                cur.execute("""
                    INSERT INTO PaymentTermItems (
                        Id, Name, Gap, Percentage, DueDate, PaymentTermId, MilestoneGapType,
                        CreatedBy, CreationDate, IsDeleted, embed_Name, embed_MilestoneGapType, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], row2['Name'], int(row2['Gap']), float(row2['Percentage']),
                    row2['DueDate'], int(row2['PaymentTermId']), row2['MilestoneGapType'],
                    row2['CreatedBy'], row2['CreationDate'], bool(row2['IsDeleted']),
                    e_name, e_gaptyp, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)

# ProjectPaymentPlans

In [51]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS ProjectPaymentPlans (
    Id            INTEGER PRIMARY KEY,
    ProjectId     INTEGER,
    PaymentTermId INTEGER,
    CreatedBy     TEXT,
    CreationDate  TEXT,
    IsDeleted     BOOLEAN,
    row_text      TEXT,
    embedding     VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "ProjectPaymentPlans.json")
file_name = "ProjectPaymentPlans"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                cur.execute("""
                    INSERT INTO ProjectPaymentPlans (
                        Id, ProjectId, PaymentTermId, CreatedBy, CreationDate, IsDeleted, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], int(row2['ProjectId']), int(row2['PaymentTermId']),
                    row2['CreatedBy'], row2['CreationDate'], bool(row2['IsDeleted']), txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)

# ProjectPaymentTerms

In [52]:
text_data_path = 'text_data.json'
with open(text_data_path, 'r', encoding='utf-8-sig') as file:
                text_data = json.load(file)
                
x = cur.execute("""
CREATE TABLE IF NOT EXISTS ProjectPaymentTerms (
    Id             INTEGER PRIMARY KEY,
    Layout         TEXT,
    ProjectId      INTEGER,
    MinimumAmount  DOUBLE PRECISION,
    -- embed_* columns
    embed_Layout   VECTOR(1024),
    row_text       TEXT,
    embedding      VECTOR(1024)
);
""")

connection.commit()
connection.rollback()

full_path = os.path.join(directory, "ProjectPaymentTerms.json")
file_name = "ProjectPaymentTerms"
text_data = text_data[file_name]
if os.path.isfile(full_path):
    with open(full_path, 'r', encoding='utf-8-sig') as file:
        data = json.load(file)

        i = 0
        try:
            for row in data[file_name]:
                row2 = pre_row(row, file_name)
                txt = row2text()
                embedding = generate_embedding(txt)

                e_layout = generate_embedding(row2['Layout']) if row2.get('Layout') else None

                cur.execute("""
                    INSERT INTO ProjectPaymentTerms (
                        Id, Layout, ProjectId, MinimumAmount, embed_Layout, row_text, embedding
                    ) VALUES (%s,%s,%s,%s,%s,%s,%s)
                """, (
                    row2['Id'], row2['Layout'], int(row2['ProjectId']),
                    float(row2['MinimumAmount']), e_layout, txt, embedding
                ))

                if (i + 1) % 100 == 0:
                    connection.commit()
                i += 1

        except Exception as e:
            print(file_name)
            print(e)